# Phase 14 — Data Warehouse: Physical Implementation

This phase creates and queries the Ice Cream dimensional warehouse. The source layer does **not** contain dimensions or facts. The warehouse does.

## Warehouse tables
- `dim_date`
- `dim_customer`
- `dim_product`
- `dim_store`
- `fact_sales`

### Grain of fact_sales
**One sales transaction for one product/customer/store/date combination.**

The physical CSV files for these tables are included in this phase folder under `data/`.

## 1. Load the warehouse files into DuckDB

DuckDB can query CSV directly. We first create physical DuckDB tables from the warehouse CSV files.

In [ ]:
import duckdb

con = duckdb.connect("icecream_dw.duckdb")

con.execute("CREATE OR REPLACE TABLE dim_date AS SELECT * FROM read_csv_auto('data/dim_date.csv')")
con.execute("CREATE OR REPLACE TABLE dim_customer AS SELECT * FROM read_csv_auto('data/dim_customer.csv')")
con.execute("CREATE OR REPLACE TABLE dim_product AS SELECT * FROM read_csv_auto('data/dim_product.csv')")
con.execute("CREATE OR REPLACE TABLE dim_store AS SELECT * FROM read_csv_auto('data/dim_store.csv')")
con.execute("CREATE OR REPLACE TABLE fact_sales AS SELECT * FROM read_csv_auto('data/fact_sales.csv')")


## 2. Inspect every warehouse table

In [ ]:
for table in ["dim_date","dim_customer","dim_product","dim_store","fact_sales"]:
    print("\n---", table, "---")
    print(con.execute(f"DESCRIBE {table}").df())
    print(con.execute(f"SELECT * FROM {table} LIMIT 5").df())


## 3. Validate primary/foreign-key relationships

In [ ]:
checks = {
    "orphan_customer": "SELECT COUNT(*) AS n FROM fact_sales f LEFT JOIN dim_customer d ON f.customer_key=d.customer_key WHERE d.customer_key IS NULL",
    "orphan_product": "SELECT COUNT(*) AS n FROM fact_sales f LEFT JOIN dim_product d ON f.product_key=d.product_key WHERE d.product_key IS NULL",
    "orphan_store": "SELECT COUNT(*) AS n FROM fact_sales f LEFT JOIN dim_store d ON f.store_key=d.store_key WHERE d.store_key IS NULL",
    "orphan_date": "SELECT COUNT(*) AS n FROM fact_sales f LEFT JOIN dim_date d ON f.date_id=d.date_id WHERE d.date_id IS NULL",
}
for name,q in checks.items(): print(name, con.execute(q).fetchone()[0])


## 4. Build a joined analytical view

This is the table students will use for many business questions. It combines the fact with the dimensions.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW vw_sales_analysis AS
SELECT
    f.transaction_id,
    d.date, d.year, d.month, d.month_name, d.weekday, d.quarter,
    c.customer_id, c.customer_name, c.city AS customer_city, c.state, c.segment,
    p.product_id, p.product_name, p.category,
    s.store_id, s.store_name, s.city AS store_city,
    f.quantity, f.unit_price, f.discount_rate, f.sales_amount, f.cost_amount, f.profit
FROM fact_sales f
JOIN dim_date d ON f.date_id=d.date_id
JOIN dim_customer c ON f.customer_key=c.customer_key
JOIN dim_product p ON f.product_key=p.product_key
JOIN dim_store s ON f.store_key=s.store_key
""")
con.execute("SELECT * FROM vw_sales_analysis LIMIT 10").df()


## 5. Example: best-selling product on a date

In [ ]:
date = "2026-06-15"
con.execute(f"""
SELECT product_name, SUM(quantity) AS units, SUM(sales_amount) AS sales
FROM vw_sales_analysis
WHERE date = DATE '{date}'
GROUP BY product_name
ORDER BY units DESC, sales DESC
""").df()


## 6. Example: best-selling product by week

In [ ]:
con.execute("""
SELECT year, EXTRACT('week' FROM date) AS week_number, product_name,
       SUM(quantity) AS units, SUM(sales_amount) AS sales
FROM vw_sales_analysis
GROUP BY year, week_number, product_name
QUALIFY ROW_NUMBER() OVER(PARTITION BY year, week_number ORDER BY units DESC, sales DESC) = 1
ORDER BY year, week_number
""").df()


## 7. Students should now use `duckdb_business_questions.sql`

The SQL file contains a large question bank with joins across fact and dimensions, window functions, date analysis, product analysis, store analysis, zero-activity analysis, promotions, and trend analysis.